In [188]:
import hexbytes
import concurrent.futures
import sys
import threading
import json
import time
import os
import random
import logging
import signal
import requests
from pprint import pprint
from web3.logs import STRICT, IGNORE, DISCARD, WARN
import pandas as pd
import matplotlib.pyplot as plt
from decimal import Decimal
from hexbytes import HexBytes
from pathlib import Path
import plotly.graph_objects as go
from requests.exceptions import HTTPError
from datetime import datetime, timezone
from web3 import Web3
from collections import defaultdict
from web3.exceptions import TransactionNotFound, BlockNotFound
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import wraps

# === CONFIGURATION ===

ETHERSCAN_API_KEY_DICT = {
    "hearthquake": {
        "INFURA_URL": "https://mainnet.infura.io/v3/d28e1ba493794e6aadf097e4964279fb",
        "ETHERSCAN_API_KEY": "d28e1ba493794e6aadf097e4964279fb",
    },
    "opensee": {
        "INFURA_URL": "https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19",
        "ETHERSCAN_API_KEY": "F7K9BTHSSB9EQT9WEGHMG3VFJ54KA8RM1K",
    },
}

INFURA_URL = ETHERSCAN_API_KEY_DICT["opensee"]["INFURA_URL"]
ETHERSCAN_API_KEY = ETHERSCAN_API_KEY_DICT["opensee"]["ETHERSCAN_API_KEY"]
CONTRACT_ADDRESS = POOL_ADDRESS = "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD"
ABI_FILE = "WETH_WBTC_pool.json"  # Load your contract ABI file
BLOCKS_FILE = "blocks_data.json"
TRANSACTIONS_FILE = "transactions.json"
METADATA_FILE = "processed_blocks.json"
BATCH_SIZE = 1000  # Number of transactions to process before writing to disk

# === CONNECT TO ETHEREUM NODE ===
w3 = Web3(
    Web3.HTTPProvider(INFURA_URL)
    #Web3.HTTPProvider("http://127.0.0.1:8545")
)

assert w3.is_connected(), "Web3 provider connection failed"
w3.eth.get_block('latest').number

23407114

In [186]:
# --------------------
# Helper Function: Get ABI from Etherscan or Disk
# --------------------

def get_abi(contract_address: str, api_key: str) -> list:
    """
    Retrieves the ABI for a given contract address.
    Checks if the ABI is available in the local 'ABI' folder.
    If not, it fetches the ABI from Etherscan using the provided API key,
    then saves it to disk for future use.
    
    Parameters:
        contract_address (str): The contract address (checksum not required here).
        api_key (str): Your Etherscan API key.
        
    Returns:
        list: The ABI loaded as a Python list.
    """
    # Ensure the ABI folder exists.
    abi_folder = "ABI"
    if not os.path.exists(abi_folder):
        os.makedirs(abi_folder)
    
    # Save ABI with filename based on contract address.
    filename = os.path.join(abi_folder, f"{contract_address}.json")
    
    # If file exists, load and return the ABI.
    if os.path.exists(filename):
        with open(filename, "r") as file:
            abi = json.load(file)
    else:
        # Construct the Etherscan API URL.
        url = f"https://api.etherscan.io/api?module=contract&action=getabi&address={contract_address}&apikey={api_key}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data["status"] == "1":
                # Parse the ABI and save it for later use.
                abi = json.loads(data["result"])
                with open(filename, "w") as file:
                    json.dump(abi, file)
            else:
                raise Exception(f"Error fetching ABI for contract {contract_address}: {data['result']}")
        else:
            raise Exception("Error connecting to the Etherscan API. Status code: " + str(response.status_code))
    return abi

# -----------------------
# Helper: Convert event to dict
# -----------------------
def event_to_dict(event):
    d = dict(event)
    if "args" in d:
        d["args"] = dict(d["args"])
    if "transactionHash" in d:
        d["transactionHash"] = d["transactionHash"].hex()
    if "blockHash" in d:
        d["blockHash"] = d["blockHash"].hex()
    return d


def get_contract_creation_block_etherscan(contract_address: str, etherscan_api_key: str) -> int:
    """
    Retrieves the contract creation block from Etherscan.
    Returns the block number as an integer.
    """
    url = (f"https://api.etherscan.io/api?module=contract&action=getcontractcreation"
           f"&contractaddresses={contract_address}&apikey={etherscan_api_key}")
    response = requests.get(url)
    data = response.json()

    if data.get("status") == "1":
        results = data.get("result", [])
        if results and len(results) > 0:
            return int(results[0]["blockNumber"])
        else:
            raise Exception("No contract creation data found.")
    else:
        raise Exception("Error fetching creation block: " + data.get("result", "Unknown error"))

# -----------------------
# Used to find at which block 1 contract has been deployed
# Might be useful later, put it in JSON in the end
# -----------------------

def get_contract_creation_block_custom(start_block=0, end_block=100000):

    def get_contract_deployments(start_block, end_block, max_workers=8):
        deployments = []

        def process_block(block_number):
            block = w3.eth.get_block(block_number, full_transactions=True)
            block_deployments = []
            for tx in block.transactions:
                if tx.to is None:
                    try:
                        receipt = w3.eth.get_transaction_receipt(tx.hash)
                        contract_address = receipt.contractAddress
                        if contract_address:
                            block_deployments.append(
                                {
                                    "block_number": block_number,
                                    "contract_address": contract_address,
                                }
                            )
                    except:
                        print(tx.hash)
            return block_deployments

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_block = {
                executor.submit(process_block, bn): bn
                for bn in range(start_block, end_block + 1)
            }
            for future in as_completed(future_to_block):
                block_deployments = future.result()
                deployments.extend(block_deployments)

        return deployments

    deployments = get_contract_deployments(start_block, end_block)

    # Save the results to a JSON file
    with open("contract_deployments.json", "w") as f:
        json.dump(deployments, f, indent=4)


# -- Step 2: Reconstruct an Event’s Signature --
def get_event_signature(event_name: str, abi: list) -> str:
    """
    Given an event name and an ABI, find the event definition and reconstruct its signature.
    For example, for event Transfer(address,address,uint256) this returns its keccak256 hash.
    """
    from eth_utils import keccak, encode_hex

    for item in abi:
        if item.get("type") == "event" and item.get("name") == event_name:
            # Build the signature string: "Transfer(address,address,uint256)"
            types = ",".join([inp["type"] for inp in item.get("inputs", [])])
            signature = f"{event_name}({types})"
            return encode_hex(keccak(text=signature))
    raise ValueError(f"Event {event_name} not found in ABI.")


def block_to_utc(block_number):
    """
    Convert a block number into its UTC timestamp.

    Parameters:
        w3 (Web3): A Web3 instance
        block_number (int): The block number

    Returns:
        datetime: The block timestamp in UTC
    """
    block = w3.eth.get_block(block_number)
    timestamp = block["timestamp"]
    return datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat()


def read_and_sort_jsonl(file_path):
    """
    Reads a JSONL file, each line being a JSON object with a field `blockNumber`,
    and returns a list of those objects sorted by blockNumber (ascending).
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                # Handle bad JSON if needed, e.g., log or skip
                print(line)
                print(f"Skipping bad JSON line: {e}")
                continue
            # Optionally, you could check that 'blockNumber' exists, is int, etc.
            if "blockNumber" not in obj:
                print(f"Skipping line with no blockNumber: {obj}")
                continue
            data.append(obj)
    # Now sort by blockNumber ascending
    # If blockNumber in file is already int, fine; else convert
    sorted_data = sorted(data, key=lambda o: int(o["blockNumber"]))
    return sorted_data


def get_address_abi_contract(contract_address, etherscan_api_key=ETHERSCAN_API_KEY):
    address = w3.to_checksum_address(contract_address)
    contract_abi = get_abi(address, etherscan_api_key)
    contract = w3.eth.contract(address=contract_address, abi=contract_abi)

    return address, contract_abi, contract

# Find the amount of token depending on the contract at the very specific block_number
# but it use ETHERSCAN API (to go further: explorer the reconstruct from all the Transfer event but slow)
# Not super useful for the moment
def get_erc20_balance_at_block(user_address, token_address, block_number):
    """
    Query ERC-20 balance of an address at a specific block.

    Parameters:
        user_address: string, account to check
        token_address: Web3 contract instance for the ERC-20 token
        block_number: int, historical block

    Returns:
        int: token balance
        None if contract is a proxy
    """
    token_address, token_abi, token_contract = get_address_abi_contract(token_address)
    user_address =  w3.to_checksum_address(user_address)
    token_name =  None
    token_symbol = None 
    try:
        token_name = token_contract.functions.name().call()
        token_symbol = token_contract.functions.symbol().call()
    except Exception as e:
        print(f"Error {e}")
        print(f"{token_address}")
        return None
    balance = token_contract.functions.balanceOf(user_address).call(
        block_identifier=block_number
    )
    print(
        f"Address {user_address} had {w3.from_wei(balance, "ether")} of {token_symbol} at block {block_number}"
    )
    return balance

def get_token_name_by_contract(token_address):
    token_address, token_abi, token_contract = get_address_abi_contract(token_address)
    return token_contract.functions.name().call()

In [121]:
user_address = "0xe2dFC8F41DB4169A24e7B44095b9E92E20Ed57eD"
token_address = "0x514910771AF9Ca656af840dff83E8264EcF986CA"
block_number = 23405236

balance = get_erc20_balance_at_block(user_address, token_address, block_number)

Address 0xe2dFC8F41DB4169A24e7B44095b9E92E20Ed57eD had 6.74577 of LINK at block 23405236


In [ ]:
sorted_transactions = read_and_sort_jsonl("new_found_tx.jsonl")
DISTINCT_PROVIDER = set()
D_BLOCK_TOTAL_LIQUIDITY_BY_CONTRACT_BY_BLOCK = {}
L_LIQUIDITY_BOOK = []
L_UNI_LIQUIDITY = []


def analyze_transaction(transaction):
    block = transaction["blockNumber"]
    event = transaction["event"]
    event_args = transaction["args"]
    contract = transaction["contract"]
    tx_hash = transaction["txHash"]
    # Transaction data in case we need to investigate  further (but we could directly store them in jsonl from the sniffer)
    # transaction_information = w3.eth.get_transaction(transaction["txHash"])
    # d_transaction_information = event_to_dict(transaction_information)
    # In case we need ABI and retrieve token information
    # token_address, contract_abi, contract = get_address_abi_contract(
    #    transaction["contract"]
    # )

    # Event part
    if event == "AddLiquidity":
        provider = event_args["provider"]
        token_amount = event_args["token_amount"]
        eth_amount = event_args["eth_amount"]
        # Provider are liquidity participant, we track all of them to count
        DISTINCT_PROVIDER.add(provider)
        L_LIQUIDITY_BOOK.append(
            {
                "block": block,
                "contract": contract,
                "provider": provider,
                "token_amount": w3.from_wei(token_amount, "ether"),
                "eth_amount": w3.from_wei(eth_amount, "ether"),
            }
        )
    elif event == "RemoveLiquidity":
        provider = event_args["provider"]
        token_amount = event_args["token_amount"]
        eth_amount = event_args["eth_amount"]
        L_LIQUIDITY_BOOK.append(
            {
                "block": block,
                "contract": contract,
                "provider": provider,
                "token_amount": -w3.from_wei(token_amount, "ether"),
                "eth_amount": -w3.from_wei(eth_amount, "ether"),
            }
        )

    elif event == "Transfer":
        _from = event_args["_from"]
        _to = event_args["_to"]
        _value = event_args["_value"]  # UniswapV1-TOKEN-Liquidity-debt
        if _from == w3.to_checksum_address(
            "0x0000000000000000000000000000000000000000"
        ):
            # We Mint Liquidity
            D_BLOCK_TOTAL_LIQUIDITY_BY_CONTRACT_BY_BLOCK[contract][
                block
            ] += w3.from_wei(_value, "ether")
            L_UNI_LIQUIDITY.append(
                {
                    "block": block,
                    "contract": contract,
                    "provider": _to,
                    "value": w3.from_wei(_value, "ether"),
                }
            )

        if _to == w3.to_checksum_address(
            w3.to_checksum_address("0x0000000000000000000000000000000000000000")
        ):
            # We Burn Liquidity
            D_BLOCK_TOTAL_LIQUIDITY_BY_CONTRACT_BY_BLOCK[contract][
                block
            ] -= w3.from_wei(_value, "ether")
            L_UNI_LIQUIDITY.append(
                {
                    "block": block,
                    "contract": contract,
                    "provider": _from,
                    "value": -w3.from_wei(_value, "ether"),
                }
            )
    # Pursechase will be used to compute SWAP (Volume, Fees) Later
    elif event == "TokenPurchase":
        buyer = event_args["buyer"]
        eth_sold = event_args["eth_sold"]
        tokens_bought = event_args["tokens_bought"]
    elif event == "EthPurchase":
        buyer = event_args["buyer"]
        tokens_sold = event_args["tokens_sold"]
        eth_bought = event_args["eth_bought"]
    # Not that useful for the moment
    elif event == "Approval":
        _owner = event_args["_owner"]
        _spender = event_args["_spender"]
        _value = event_args["_value"]
    else:
        print(f"Event not Known: {event}")


result = []
print(len(sorted_transactions))
for tx in sorted_transactions:
    # print(tx)
    analyze_transaction(tx)

print(len(DISTINCT_PROVIDER))
pprint(D_BLOCK_TOTAL_LIQUIDITY_BY_CONTRACT_BY_BLOCK)

45092
388
{}


In [91]:
print(len(L_LIQUIDITY_BOOK) + len(L_UNI_LIQUIDITY))
df_liquidity_book = pd.DataFrame(L_LIQUIDITY_BOOK)
df_uni_liquidity =pd.DataFrame(L_UNI_LIQUIDITY)
df = pd.concat([df_liquidity_book, df_uni_liquidity], ignore_index=False)

1608


In [92]:
import pandas as pd
import plotly.express as px

# Ensure df is sorted by block
df = df.sort_values("block").copy()

# Vectorized cumulative unique provider counts
# Create a column combining contract + provider for uniqueness
df["contract_provider"] = df["contract"] + "_" + df["provider"]

# Drop duplicates for cumulative tracking per contract
df_unique = df.drop_duplicates(subset=["contract_provider", "block"]).copy()
df_unique.loc[:, "count"] = 1  # avoid SettingWithCopyWarning

# Deduce contracts dynamically from the df
contracts = df_unique["contract"].unique()

# Pivot to have blocks as index, contracts as columns
pivot = df_unique.pivot_table(
    index="block", columns="contract", values="count", aggfunc="sum", fill_value=0
)

# Compute cumulative sum per contract
cumulative = pivot.cumsum().reset_index()

# Melt for Plotly long format
cumulative_long = cumulative.melt(
    id_vars="block",
    value_vars=contracts,
    var_name="contract",
    value_name="cumulative_providers",
)

# Plot interactive stacked area chart
fig = px.area(
    cumulative_long,
    x="block",
    y="cumulative_providers",
    color="contract",
    labels={"cumulative_providers": "Cumulative Unique Providers", "block": "Block"},
)

fig.update_traces(mode="lines", hoverinfo="x+y+name", opacity=0.6)

# Interactive legend toggle
fig.update_layout(
    title="Cumulative Unique Providers per Contract by Block",
    hovermode="x unified",
    legend_title_text="Contract",
    legend=dict(itemclick="toggle", itemdoubleclick="toggleothers"),
)

fig.show()

In [ ]:
FULL_EVENT_BY_CONTRACTS = json.load(open(r"real/FULL_EVENT_BY_CONTRACTS.json"))
# We often need the Initial Factory informations
uniswap_factory_address, uniswap_factory_abi, uniswap_factory_contract = (
    get_address_abi_contract(Web3.to_checksum_address("0xC0A47DFE034B400B47BDAD5FECDA2621DE6C4D95"))
)  # Uniswap Genesis Factory
result = []
for exchange_address in FULL_EVENT_BY_CONTRACTS.keys():
    try:
        token_addr = uniswap_factory_contract.functions.getToken(exchange_address).call()
        token_name = get_token_name_by_contract(token_addr)
    except Exception as e:
        print(
            f"Something went wrong: {e} for: Exchange {exchange_address}, underlying token {token_addr}"
        )
        continue
    if isinstance(token_name, bytes):
        name = token_name.decode("utf-8", errors="ignore").replace("\x00", "").strip()
    else:
        name = token_name.strip()
    result.append((name, exchange_address))

print(result)

Something went wrong: ("The function 'name' was not found in this ", "contract's abi.") for: Exchange 0x48B04d2A05B6B604d8d5223Fd1984f191DED51af, underlying token 0x1985365e9f78359a9B6AD760e32412f4a445E862
Something went wrong: ("The function 'name' was not found in this ", "contract's abi.") for: Exchange 0x69f276aBd6456152d519D23086031DA7c73F91b8, underlying token 0x57Ab1E02fEE23774580C119740129eAC7081e9D3
Something went wrong: ("The function 'name' was not found in this ", "contract's abi.") for: Exchange 0x055BAb4bcC17d2AB155fff017fC5E093556C6aD2, underlying token 0xEB9951021698B42e4399f9cBb6267Aa35F82D59D
Something went wrong: Error fetching ABI for contract 0xecF3958d0f82291CA1FF6c9BDa8Eb3c50eE41cE3: Contract source code not verified for: Exchange 0x7d31fc38dDD7d6907F820F4268f1d8D5d5797826, underlying token 0xecF3958d0f82291CA1FF6c9BDa8Eb3c50eE41cE3
Something went wrong: ("The function 'name' was not found in this ", "contract's abi.") for: Exchange 0x97deC872013f6B5fB443861090ad

BadResponseFormat: The response was in an unexpected format and unable to be parsed. The "jsonrpc" field must be present with a value of "2.0". The raw response is: {'message': 'no nodes available for requested block height after filtering 6 available nodes'}

In [ ]:
len(result)
# Convert to dict (name → address)
token_dict = dict(result)

# Write to JSON file
with open("real/tokens.json", "w") as f:
    json.dump(token_dict, f, indent=4)

In [182]:
token_filter = [
    "0x2C4BD064B998838076FA341A83D007FC2FA50957",
    "0x255E60C9D597DCAA66006A904ED36424F7B26286",
    "0xE8E45431B93215566BA923a7E611B7342EA954DF",
    "0xF173214C720F58E03E194085B1DB28B50ACDEEAD",
    "0xC6581CE3A005E2801C1E0903281BBD318EC5B5C2",
    "0x494D82667C3ED3AC859CCA94B1BE65B0540EE3BB",
    "0x077D52B047735976DFDA76FEF74D4D988AC25196",
    "0xC4A1C45D5546029FD57128483AE65B56124BFA6A",
    "0x7DC095A5CF7D6208CC680FA9866F80A53911041A",
    "0x2135D193BF81ABBEAD93906166F2BE32B2492C04",
    "0x4D2F5CFBA55AE412221182D8475BC85799A5644B",
    "0x87D80DBD37E551F58680B4217B23AF6A752DA83F",
    "0x060A0D4539623B6AA28D9FC39B9D6622AD495F41",
    "0x6B4540F5EE32DDD5616C792F713435E6EE4F24AB",
    "0xB99A23B1A4585FC56D0EC3B76528C27CAD427473",
    "0x04045481B044534ED3CB1E24254B471CFADDEB3D",
    "0xC0E77CDD039A3F731AE0F5C6C9F4A91D4BC28880",
]
token_filter = [Web3.to_checksum_address(k) for k in token_filter]
FULL_EVENT_BY_CONTRACTS = {
    Web3.to_checksum_address(k): v for k, v in FULL_EVENT_BY_CONTRACTS.items()
}

subdict = {
    k: FULL_EVENT_BY_CONTRACTS[k] for k in token_filter if k in FULL_EVENT_BY_CONTRACTS
}

In [183]:
subdict

{'0x2C4Bd064b998838076fa341A83d007FC2FA50957': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {},
  'TokenPurchase': {},
  'Transfer': {}},
 '0x255e60c9d597dCAA66006A904eD36424F7B26286': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {},
  'TokenPurchase': {},
  'Transfer': {}},
 '0xe8e45431b93215566BA923a7E611B7342Ea954DF': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {},
  'TokenPurchase': {},
  'Transfer': {}},
 '0xF173214C720f58E03e194085B1DB28B50aCDeeaD': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {},
  'TokenPurchase': {},
  'Transfer': {}},
 '0xC6581Ce3A005e2801c1e0903281BBd318eC5B5C2': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {},
  'TokenPurchase': {},
  'Transfer': {}},
 '0x494d82667c3eD3ac859cCA94b1Be65b0540eE3Bb': {'AddLiquidity': {},
  'Approval': {},
  'EthPurchase': {},
  'RemoveLiquidity': {}

In [ ]:
OUTPUT_FILE = "all_found_tx.jsonl"
STATE_FILE = "new_scan_state.json"
CHUNK_SIZE = 10000
START_BLOCK = 0
END_BLOCK = 'latest'  # None = latest

# --- CONFIG: contracts and events with optional argument_filters ---
FULL_EVENT_BY_CONTRACTS = json.load(open(r"real/FULL_EVENT_BY_CONTRACTS.json"))
EVENTS = FULL_EVENT_BY_CONTRACTS
# EVENTS = {
#     "0xc0a47dFe034B400B47bDaD5FecDa2621de6c4d95":{

#     },
#     "0x255e60c9d597dCAA66006A904eD36424F7B26286": {
#         "AddLiquidity": {},
#         "RemoveLiquidity": {},
#         "Transfer": {},
#         "EthPurchase": {},
#         "TokenPurchase": {},
#         "Approval": {},
#     },
#     "0xF173214C720f58E03e194085B1DB28B50aCDeeaD": {
#         "AddLiquidity": {},
#         "RemoveLiquidity": {},
#         "Transfer": {},
#         "EthPurchase": {},
#         "TokenPurchase": {},
#         "Approval": {},
#     },
# }

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# --- Helpers ---
contracts = {}  # address -> Contract object


def init_contracts():
    for addr in EVENTS.keys():
        checksum = w3.to_checksum_address(addr)
        abi = get_abi(checksum, ETHERSCAN_API_KEY)
        contracts[checksum] = w3.eth.contract(address=checksum, abi=abi)


def load_state():
    return (
        json.load(open(STATE_FILE))
        if os.path.exists(STATE_FILE)
        else {"done_chunks": []}
    )


def save_state(state):
    json.dump(state, open(STATE_FILE, "w"))


def load_seen():
    seen = set()
    if os.path.exists(OUTPUT_FILE):
        for l in open(OUTPUT_FILE):
            try:
                seen.add(json.loads(l)["txHash"].lower())
            except:
                pass
    return seen


def append_lines(records):
    if not records:
        return
    with open(OUTPUT_FILE, "a") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


def scan_event(c, ev_name, arg_filters, start, end, seen):
    ev = getattr(c.events, ev_name)()
    backoff = 1
    while True:
        try:
            logs = ev.get_logs(
                from_block=start, to_block=end, argument_filters=arg_filters
            )
            break
        except Exception as e:
            msg = str(e).lower()
            if "429" in msg or "402" in msg:
                logging.warning(f"rate limit {msg}, retry in {backoff}s")
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)
                continue
            raise
    new = []
    for log in logs:
        tx = log["transactionHash"].hex()
        blk = log["blockNumber"]
        if tx.lower() in seen:
            continue
        seen.add(tx.lower())
        rec = {
            "contract": c.address,
            "event": ev_name,
            "txHash": tx,
            "blockNumber": blk,
            "args": dict(log["args"]),
        }
        print(f"FOUND {rec}", flush=True)
        logging.info(f"found {c.address} {ev_name} {tx}")
        new.append(rec)
    append_lines(new)


# --- Main ---
def main():
    init_contracts()
    latest = w3.eth.block_number if END_BLOCK is None else END_BLOCK
    state = load_state()
    done_chunks = {tuple(x) for x in state["done_chunks"]}
    seen = load_seen()
    start = START_BLOCK
    stop = False

    def handle_sig(sig, frame):
        nonlocal stop
        stop = True
        logging.info("Stopping after current chunk...")

    signal.signal(signal.SIGINT, handle_sig)

    while start <= latest and not stop:
        end = min(start + CHUNK_SIZE - 1, latest)
        key = (start, end)
        if key in done_chunks:
            start = end + 1
            continue
        logging.info(f"Scanning blocks {start}-{end}")
        all_ok = True
        for addr, ev_map in EVENTS.items():
            c = contracts[w3.to_checksum_address(addr)]
            for ev_name, arg_filters in ev_map.items():
                # remove None values from argument_filters
                arg_filters_clean = {
                    k: v for k, v in arg_filters.items() if v is not None
                }
                try:
                    scan_event(c, ev_name, arg_filters_clean, start, end, seen)
                except Exception as e:
                    logging.warning(
                        f"Error scanning {addr} {ev_name} {start}-{end}: {e}"
                    )
                    all_ok = False
        if all_ok:
            state["done_chunks"].append(key)
            save_state(state)
            logging.info(f"Chunk {start}-{end} done")
        start = end + 1

    logging.info("Finished scanning.")


main()

2025-09-20 13:18:49,425 INFO Scanning blocks 0-9999
2025-09-20 13:18:55,897 INFO Chunk 0-9999 done
2025-09-20 13:18:55,898 INFO Scanning blocks 10000-19999
2025-09-20 13:19:02,504 INFO Chunk 10000-19999 done
2025-09-20 13:19:02,505 INFO Scanning blocks 20000-29999
2025-09-20 13:19:06,349 INFO Chunk 20000-29999 done
2025-09-20 13:19:06,349 INFO Scanning blocks 30000-39999
2025-09-20 13:19:10,912 INFO Chunk 30000-39999 done
2025-09-20 13:19:10,912 INFO Scanning blocks 40000-49999
2025-09-20 13:19:14,252 WARNING rate limit 429 client error: too many requests for url: https://mainnet.infura.io/v3/d28e1ba493794e6aadf097e4964279fb, retry in 1s
2025-09-20 13:19:18,436 WARNING rate limit 429 client error: too many requests for url: https://mainnet.infura.io/v3/d28e1ba493794e6aadf097e4964279fb, retry in 1s
2025-09-20 13:19:21,665 INFO Chunk 40000-49999 done
2025-09-20 13:19:21,666 INFO Scanning blocks 50000-59999
2025-09-20 13:19:25,860 WARNING rate limit 429 client error: too many requests for

In [124]:
# PIECE OF CODE TO GET THE UNIV1 EXCHANGE ADDRESS

### We focus on the Factory contract of uniswap v1 "0xc0a47dFe034B400B47bDaD5FecDa2621de6c4d95"
# We need the ProviderNode to be initialized already
# Why debug_traceTransaction Is the Best Option
# It replays the transaction within the exact historical state using your archive node and returns a detailed call graph, including internal calls and value flows—not just high-level transfers. You can choose tracers like callTracer (which outputs call frames and nested structure) for highest clarity and insight.
# Alternatives like event logs or transaction receipts won't capture internal calls, since those are not emitted as events. You need a trace API to follow what's happening inside smart contract execution.
uniswap_v1_factory_address = w3.to_checksum_address("0xc0a47dFe034B400B47bDaD5FecDa2621de6c4d95")
uniswap_v1_factory_abi = get_abi(uniswap_v1_factory_address, ETHERSCAN_API_KEY)
uniswap_v1_factory_contract = w3.eth.contract(
    address=uniswap_v1_factory_address, abi=uniswap_v1_factory_abi
)

def trace_internal_transactions(tx_hash: str, tracer: str = "callTracer") -> dict:
    """
    Performs debug_traceTransaction with specified tracer (default: callTracer).
    Returns the full trace result as a Python dict.
    """
    trace = w3.provider.make_request(
        "debug_traceTransaction", [tx_hash, {"tracer": tracer}]
    )
    return trace.get("result", {})


def extract_internal_transfers_from_trace(trace: dict) -> list:
    """
    Recursively traverses the 'calls' in the trace to gather internal transfers.
    Returns list of dicts with from, to, value, gasUsed, etc.
    """
    transfers = []

    def recurse(call):
        # Internal transfer if value is non-zero
        value = int(call.get("value", "0x0"), 16)
        if value > 0:
            transfers.append(
                {
                    "from": call.get("from"),
                    "to": call.get("to"),
                    "value": value,
                    "gas": int(call.get("gas", "0x0"), 16),
                    "gasUsed": int(call.get("gasUsed", "0x0"), 16),
                    "type": call.get("type"),
                    "error": call.get("error"),
                }
            )
        for sub in call.get("calls", []) or []:
            recurse(sub)

    recurse(trace)
    return transfers


def get_internal_transactions_for_contract(
    contract_address: str, from_block: int, to_block: int
):
    """Scan blocks, identify txs to/from contract, and trace internal calls."""
    results = []
    for block_num in range(from_block, to_block + 1):
        block = w3.eth.get_block(block_num, full_transactions=True)
        for tx in block.transactions:
            if (
                 tx["to"]
                 and tx["to"] == uniswap_v1_factory_address
                 or tx["from"] == uniswap_v1_factory_address
            ):

                function, params = uniswap_v1_factory_contract.decode_function_input(
                    tx["input"]
                )
                if function.fn_name == 'createExchange':
                    #print(tx)
                    # print('Called function:', function.fn_name)
                    #print('With arguments:', params)
                    uni_created_token = params['token']
                    univ1_token_address = w3.to_checksum_address(uni_created_token)
                    univ1_factory_abi = get_abi(univ1_token_address, ETHERSCAN_API_KEY)
                    univ1_factory_contract = w3.eth.contract(
                        address=univ1_token_address, abi=univ1_factory_abi
                    )
                    token_name =  None
                    token_symbol = None 
                    try:
                        token_name = univ1_factory_contract.functions.name().call()
                        token_symbol = univ1_factory_contract.functions.symbol().call()
                        print(f"Token Name: {token_name}")
                        print(f"Token Symbol: {token_symbol}")
                    except:
                        print(f"Contract is a proxy {uni_created_token}")

                    token_created_exchange_address = uniswap_v1_factory_contract.functions.getExchange(
                        uni_created_token
                    ).call()
                    print(f"Token {token_name} UniExchange_address: {token_created_exchange_address}")
                    a = w3.to_checksum_address(token_created_exchange_address)
                    b = get_abi(a, ETHERSCAN_API_KEY)
                    c = w3.eth.contract(
                        address=a, abi=b
                    )
                    try:
                        #print(f"{c.functions.name.call()}")
                        #print(f"{c.functions.symbol.call()}")
                        print(f"Token Address: {c.functions.tokenAddress.call()}")
                        #print(
                        #    f"Is this the same ? {c.functions.tokenAddress.call() == uni_created_token}"
                        #)
                    except:
                        print(f"proxy: {c}")
            #     tx_hash = tx.hash.hex()
            #     trace = trace_internal_transactions(tx_hash)
            #     transfers = extract_internal_transfers_from_trace(trace)
            #     results.append(
            #         {
            #             "tx_hash": tx_hash,
            #             "block": block_num,
            #             "internal_transfers": transfers,
            #         }
            #     )
    return results

internal_txs = get_internal_transactions_for_contract(
    uniswap_v1_factory_contract, 6627900, w3.eth.get_block("latest").number
    #uniswap_v1_factory_contract, 6500000, w3.eth.get_block("latest").number
)
for entry in internal_txs:
    print(entry["tx_hash"], entry["internal_transfers"])

Token Name: b'Maker\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Token Symbol: b'MKR\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
Token b'Maker\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00' UniExchange_address: 0x2C4Bd064b998838076fa341A83d007FC2FA50957
Token Address: 0x9f8F72aA9304c8B593d555F12eF6589cC3A579A2
Token Name: OMGToken
Token Symbol: OMG
Token OMGToken UniExchange_address: 0xDdee242662323a3cff3F9AA139fFA496aC3C73B0
Token Address: 0xd26114cd6EE289AccF82350c8d8487fedB8A0C07
Token Name: 0x Protocol Token
Token Symbol: ZRX
Token 0x Protocol Token UniExchange_address: 0xaE76c84C9262Cdb9abc0C2c8888e62Db8E22A0bF
Token Address: 0xE41d2489571d322189246DaFA5ebDe1F4699F498
Token Name: BNB
Token Symbol: BNB
Token BNB UniExchange_address: 0x255e60c9d597dCAA66006A904eD36424F7B26286
Token Addre

HTTPError: 429 Client Error: Too Many Requests for url: https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19

In [ ]:
# Super important code, for 1 transaction, we get all the Transfer event and we analyze which token has been exchanged
# we also get the Gas + ETH send. If we analyze all the transaction from 1 exchange, we can probably deduct all the liquidity
# token issued by the pair_exchange
uniswap_v1_BNB_exchange_address = w3.to_checksum_address(
    "0x255e60c9d597dCAA66006A904eD36424F7B26286"
)
uniswap_v1_BNB_exchange_abi = get_abi(
    uniswap_v1_BNB_exchange_address, ETHERSCAN_API_KEY
)
uniswap_v1_BNB_exchange_contract = w3.eth.contract(
    address=uniswap_v1_BNB_exchange_address, abi=uniswap_v1_BNB_exchange_abi
)
transaction = w3.eth.get_transaction(
    "1b53439a36b357c712a4abe860607c6e4d88a002dd26f97244a8ef3208b2f8b6"
)
d_transaction = event_to_dict(transaction)
tx_eth_value = w3.from_wei(d_transaction["value"], "ether")
decoded = uniswap_v1_BNB_exchange_contract.events.Transfer().process_receipt(
    w3.eth.get_transaction_receipt(transaction.hash),
    DISCARD,
)
for ev in decoded:
    d_ev = event_to_dict(ev)
    d_from = d_ev['args']['_from']
    d_to = d_ev['args']['_to']
    d_value = w3.from_wei(d_ev["args"]["_value"], "ether")
    d_address = d_ev['address']
    d_block = d_ev['blockNumber']
    d_tx_hash = d_ev["transactionHash"]
    token_address = w3.to_checksum_address(d_address)
    token_abi = get_abi(token_address, ETHERSCAN_API_KEY)
    token_contract = w3.eth.contract(address=token_address, abi=token_abi)
    symbol = token_contract.functions.symbol().call()
    decimals = token_contract.functions.decimals().call()
    print(
        f"Block number {d_block}, {tx_eth_value} ETH was used, {d_to} received {d_value} of {symbol} from {d_from} (tx_hash is {d_tx_hash})"
    )

def analyze_transaction_transfers(tx_hash, pair_exchange_contract, etherscan_api_key):
    result = []
    transaction = w3.eth.get_transaction(
        tx_hash
    )
    d_transaction = event_to_dict(transaction)
    tx_eth_value = w3.from_wei(d_transaction["value"], "ether")
    decoded = pair_exchange_contract.events.Transfer().process_receipt(
        w3.eth.get_transaction_receipt(transaction.hash),
        DISCARD,
    )
    for ev in decoded:
        _ = {}
        d_ev = event_to_dict(ev)
        d_from = d_ev["args"]["_from"]
        d_to = d_ev["args"]["_to"]
        d_value = w3.from_wei(d_ev["args"]["_value"], "ether")
        d_address = d_ev["address"]
        d_block = d_ev["blockNumber"]
        d_tx_hash = d_ev["transactionHash"]
        token_address = w3.to_checksum_address(d_address)
        token_abi = get_abi(token_address, etherscan_api_key)
        token_contract = w3.eth.contract(address=token_address, abi=token_abi)
        symbol = token_contract.functions.symbol().call()
        decimals = token_contract.functions.decimals().call()
        _["d_block"] = d_block
        _["d_from"] = d_from
        _["d_to"] = d_to
        _["d_value"] = d_value
        _["d_address"] = d_address
        _["d_tx_hash"] = d_tx_hash
        _["symbol"] = symbol
        _["decimals"] = decimals
        print(
            f"Block number {d_block}, {tx_eth_value} ETH was used, {d_to} received {d_value} of {symbol} from {d_from} (tx_hash is {d_tx_hash})"
        )
        result.append(_)
    return result

In [123]:
# Copy of the exploration block to trace function block by block
# Super interesting to get all the liquidity
# The problem is, we need the receipt of the transaction, and also its not straightforward to see
# the amount of liquidity directly from those call

uniswap_v1_factory_address, uniswap_v1_factory_abi, uniswap_v1_factory_contract = (
    get_address_abi_contract("0x255e60c9d597dCAA66006A904eD36424F7B26286")
)
def get_internal_transactions_for_contract(
    contract_address: str, from_block: int, to_block: int
):
    """Scan blocks, identify txs to/from contract, and trace internal calls."""
    results = []
    for block_num in range(from_block, to_block + 1):
        block = w3.eth.get_block(block_num, full_transactions=True)
        for tx in block.transactions:
            if (
                tx["to"]
                and (tx["to"] == uniswap_v1_factory_address
                or tx["from"] == uniswap_v1_factory_address)
            ):
                function, params = uniswap_v1_factory_contract.decode_function_input(
                    tx["input"]
                )
                if function.fn_name == "createExchange":
                    print(tx)
                    print('Called function:', function.fn_name)
                    print('With arguments:', params)
                    # print(f"Deadline: {block_to_utc(params['deadline'])}")
                    print(f"Deadline: {datetime.fromtimestamp(params['deadline'], tz=timezone.utc)}")
                    print(f"Value: {w3.from_wei(tx.value, 'ether')}")
                    try:
                        print(f"Receipt from transaction: {w3.eth.get_transaction_receipt(tx.hash)}")
                    except:
                        print(f"Can't find 0x{tx.hash.hex()}")
                    # tx.hash.hex()
    return results

# Interesting but I'm pruned
# trace = w3.provider.make_request("trace_transaction", [tx_hash])
# print(trace)
block_1 = 6845140
block_2 = 6850000
internal_txs = get_internal_transactions_for_contract(
    uniswap_v1_factory_contract,
    block_1,
    block_2
    #w3.eth.get_block("latest").number,
)

KeyboardInterrupt: 

In [ ]:
# Very old piece of code, just interesting for getting the signature of event
# Let's keep it in case of

# Example: get the Transfer event signature.
transfer_sig = get_event_signature("Transfer", token_abi)
add_liq_sig = get_event_signature("AddLiquidity", token_abi)
remove_liq_sig = get_event_signature("RemoveLiquidity", token_abi)
print("Transfer signature hash:", transfer_sig)

# -- Step 3: Determine Token Genesis Block and Set Starting Block --
# Assume you have a helper function get_contract_creation_block() that returns the creation block number.
try:
    genesis_block = get_contract_creation_block(token_address, ETHERSCAN_API_KEY)
    start_block = max(genesis_block - 1, 0)
except Exception as e:
    print("Error retrieving genesis block, defaulting to block 0:", e)
    start_block = 0

# -- Step 4: Fetch Transfer Events and Dump to a File (JSON Serializing) --
def get_transfer_events_paginated(token_contract, from_block: int, to_block: int, chunk_size: int = 5000, max_workers: int = 1) -> list:
    """
    Fetches Transfer events for a token_contract in the block range [from_block, to_block],
    paginating by chunk_size to avoid Infura's result limit. Uses moderate parallelization.

    Args:
        token_contract: A Web3 contract instance with a loaded ABI.
        from_block (int): The starting block number.
        to_block (int): The ending block number.
        chunk_size (int): How many blocks to query per chunk (default 5000).
        max_workers (int): Maximum number of parallel workers (default 4).

    Returns:
        List of events.
    """
    events_collected = []
    block_ranges = []
    
    # Divide the full range into chunks.
    for start_blk in range(from_block, to_block + 1, chunk_size):
        end_blk = min(start_blk + chunk_size - 1, to_block)
        block_ranges.append((start_blk, end_blk))
    
    def fetch_range(brange):
        print(f"Fetching for {brange}")
        start_blk, end_blk = brange
        attempts = 0
        max_retries = 5
        while attempts < max_retries:
            try:
                # Add delay to mitigate rate limits.
                time.sleep(random.uniform(1, 3))
                #events = token_contract.events.Transfer.get_logs(from_block=start_blk, to_block=end_blk)
                events = token_contract.events.AddLiquidity.get_logs(from_block=start_blk, to_block=end_blk)
                print(len(events))
                return events
            except Exception as e:
                if "429" in str(e):
                    sleep_time = random.uniform(1, 5)
                    print(f"429 error for blocks {start_blk}-{end_blk}: retrying after {sleep_time:.2f} seconds...")
                    time.sleep(sleep_time)
                    attempts += 1
                else:
                    print(f"Error fetching logs for blocks {start_blk}-{end_blk}: {e}")
                    return []
        return []  # Return empty list if all retries fail.
    
    # Use moderate parallelization.
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_range = {executor.submit(fetch_range, brange): brange for brange in block_ranges}
        for future in concurrent.futures.as_completed(future_to_range):
            events = future.result()
            events_collected.extend(events)
    
    return events_collected
    
# Custom function to convert a web3 event (and its custom types) to a plain dict.
def serialize_event(event):
    # Convert the AttributeDict to a normal dict.
    event_dict = dict(event)
    # Ensure all values are JSON serializable (convert any bytes, HexBytes etc. to a string)
    for key, value in event_dict.items():
        if hasattr(value, "hex"):
            event_dict[key] = value.hex()
    # Also convert inner "args" if present.
    if "args" in event_dict:
        args = dict(event_dict["args"])
        for k, v in args.items():
            if hasattr(v, "hex"):
                args[k] = v.hex()
        event_dict["args"] = args
    return event_dict

# Fetch logs from start_block to the current block (latest)
latest_block = w3.eth.block_number


event_list = get_transfer_events_paginated(token_contract, start_block, latest_block)
# Dump the result to a file (pretty-printing the JSON)
output_filename = f"transfer_events_{contract_address}.json"
with open(output_filename, "w") as f:
    import json
    json.dump(serialized_events, f, indent=4)

print(f"Dumped {len(serialized_events)} events to {output_filename}")


In [ ]:
# This piece of code explore block 1 by 1 to find every transaction emitted or received from 1 address
# It's super slow but very deep investigation

def get_internal_transactions_with_trace(
    contract_address: str,
    from_block: int,
    to_block: int,
    max_workers: int = 16,
):
    """
    Fetch all transactions involving a contract and optionally trace internal calls.

    Parameters:
        w3: Web3 instance connected to a local archive node.
        contract_address: Ethereum contract address (string).
        from_block: Starting block number (int).
        to_block: Ending block number (int).
        max_workers: Number of threads for parallel fetching.

    Returns:
        List of dictionaries, each containing:
            - 'transaction': The transaction object.
            - 'internal_calls': List of internal calls (empty if trace not available).
    """
    results = []

    def process_block(block_number: int):
        """Fetch transactions in a block and trace internal calls if available."""
        block = w3.eth.get_block(block_number, full_transactions=True)
        block_results = []

        for tx in block.transactions:
            # Filter top-level transactions to/from the contract
            if tx["to"] == contract_address or tx["from"] == contract_address:
                tx_entry = {"transaction": tx, "internal_calls": []}

                # Try to fetch internal calls via trace_transaction
                try:
                    trace = w3.manager.request_blocking(
                        "trace_transaction", [tx["hash"].hex()]
                    )
                    tx_entry["internal_calls"] = trace
                except Exception as e:
                    # If trace API not enabled, just skip internal calls
                    tx_entry["internal_calls"] = []

                block_results.append(tx_entry)

        return block_results

    # Use ThreadPoolExecutor to fetch blocks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_block, b): b
            for b in range(from_block, to_block + 1)
        }
        for future in as_completed(futures):
            results.extend(future.result())

    return results

contract_address = Web3.toChecksumAddress("0x255e60c9d597dCAA66006A904eD36424F7B26286")
from_block = 6845140
to_block = 6850000
txs_with_internal = get_internal_transactions_with_trace(
    contract_address, from_block, to_block
)
for tx_entry in txs_with_internal:
    print(tx_entry)

In [ ]:
# Important code
# We look for the Genesis Uniswap factory, and we get all its events (Only 1 for the factory: 'NewExchange')
# Then we scan from 0 to latest block every NexEchange created from this Factory
# (We have the filter of events in case we are filtering events from contract that have multiple events to remove when we don't care)
address, abi, contract = get_address_abi_contract("0xC0A47DFE034B400B47BDAD5FECDA2621DE6C4D95") # Uniswap Genesis Factory
start_block = 0
end_block = 'latest'
# list all event names
event_names = [ev.event_name for ev in contract.events]
print(event_names)

# define which events you want and filters directly
events_to_scan = [
    contract.events.NewExchange().get_logs,
    #contract.events.Transfer().get_logs,
    #contract.events.Approval().get_logs,
]
L_LOGS = [] # IMPORTANT
for get_logs_fn in events_to_scan:
    logs = get_logs_fn(
        from_block=start_block,
        to_block=end_block,
        argument_filters={},  # or {"from": some_address}, {"to": [addr1, addr2]}
    )
    for log in logs:
        # print(log["transactionHash"].hex(), log["blockNumber"], log["event"])
        L_LOGS.append(log)

# Important code we use in combination with the events filter
# We created a list of Exchange created by the Uniswap V1 Factory Contract and we list all their Events
# We create the Dictionnary
# "exchange_address_1": {"event_1": {}, event_2: {}, event_3:{}}
# This dict fed with the code allow us to retrieve every transactions with the events(logs) of this exchange
# we can then sniff Liquidity out of it

FULL_EVENT_BY_CONTRACTS = {}  # IMPORTANT
for i in L_LOGS:
    add, abi, contract = get_address_abi_contract(i.args.exchange)
    event_names = [ev.event_name for ev in contract.events]
    FULL_EVENT_BY_CONTRACTS[add] = {event: {} for event in event_names}
    time.sleep(1)

print(len(FULL_EVENT_BY_CONTRACTS)) # ~ 4019
filename = "real/FULL_EVENT_BY_CONTRACTS.json"
if os.path.exists(filename):
    print(f"{filename} already exists!")
else:
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(filename, f, ensure_ascii=False, indent=4)

['NewExchange']
c546ea6ddf40359322b63c19c897cf0818add5e7bc964817c392521170212dcd 6627956 NewExchange
e399afc6974b53775aa0fe9fd3bb353f2dc24c410a95b2b782a26bee7834f6f3 6627966 NewExchange
14455f1af43a52112d4ccf6043cb081fea4ea3a07d90dd57f2a9e1278114be94 6627972 NewExchange
b256d3ed140c7ce36a1434077e36105cd8bac8229e42cf169cd019a66b42d71c 6627974 NewExchange
7ad3f3ea801892805dc244002ecdce2c7a27c8db45f91ebc4db8de98bf489d72 6627976 NewExchange
504b04b5d7f847250f44200c3f2b174563a17eec67b885048f3d815700d424a5 6627979 NewExchange
ad1ddcebe7540aa5e91f6ebae4d9d031b899af23d58c53423d042e7ceda22729 6627984 NewExchange
a06ab5e584115abfe7602ee40d07e1a55d9876d6e6e80cdf90b8f46cf88df214 6627987 NewExchange
d4acf7ae57c72a2ed37665a3e13ceb880c725576336bb1324fbd7f8b759c7a7c 6627990 NewExchange
a4d14716676cc569b39e567ea3191b86cd60c1fc190e98ff5318b6a6764dac1a 6627992 NewExchange
68633c01ef59015cf64d768b8425a2fda461c8bd5166ff21e2f8da3107eb6754 6627994 NewExchange
27bb779d5cc8a7c3e9356bf48f08ed7f107140befcdec336c